# AWS

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

This notebook covers the AWS services a data engineer actually uses day to day: S3 and its storage classes, the Glue Data Catalog and Lake Formation, Glue and EMR for processing, Athena and Redshift for analytics, Kinesis, MSK, SQS, and DMS for ingestion, plus the IAM, KMS, CloudWatch, and cost mechanics that decide whether a pipeline is production-grade. Services that come up in generic AWS interviews but rarely in data work (EC2 sizing trivia, Route 53, AMI baking, Elastic Transcoder, connection draining) are deliberately omitted. Each section ends with the mistakes that cost real money or real incidents, and the *Things to remember* sections near the end compress the numbers worth memorizing.


## Contents

<details>
<summary>34 sections - click to expand</summary>

- [The AWS mental model for data engineering](#the-aws-mental-model-for-data-engineering)
- [S3 for data engineering: objects, consistency, and cost](#s3-for-data-engineering-objects-consistency-and-cost)
- [S3 storage classes and lifecycle rules](#s3-storage-classes-and-lifecycle-rules)
- [Multipart upload and large files](#multipart-upload-and-large-files)
- [Designing an AWS data lake: raw, curated, analytics](#designing-an-aws-data-lake-raw-curated-analytics)
- [Tuning S3 layout and file sizes for Athena and Glue](#tuning-s3-layout-and-file-sizes-for-athena-and-glue)
- [Governing the lake with AWS Lake Formation](#governing-the-lake-with-aws-lake-formation)
- [Glue, EMR, Lambda, or ECS: picking the compute for a job](#glue-emr-lambda-or-ecs-picking-the-compute-for-a-job)
- [Writing an AWS Glue PySpark job](#writing-an-aws-glue-pyspark-job)
- [Tuning Glue job sizing and bookmarks](#tuning-glue-job-sizing-and-bookmarks)
- [EMR fleets and Spot capacity for batch processing](#emr-fleets-and-spot-capacity-for-batch-processing)
- [Querying S3 with Athena and partition projection](#querying-s3-with-athena-and-partition-projection)
- [Tuning Athena cost and speed](#tuning-athena-cost-and-speed)
- [Amazon Redshift: architecture and when to use it](#amazon-redshift-architecture-and-when-to-use-it)
- [Redshift Spectrum for querying the lake](#redshift-spectrum-for-querying-the-lake)
- [Loading and optimizing data in Redshift](#loading-and-optimizing-data-in-redshift)
- [COPY and UNLOAD: Redshift bulk paths](#copy-and-unload-redshift-bulk-paths)
- [Kinesis, MSK, SQS, and EventBridge: how they differ](#kinesis-msk-sqs-and-eventbridge-how-they-differ)
- [Ingesting streams with Amazon Kinesis](#ingesting-streams-with-amazon-kinesis)
- [Change data capture with AWS DMS](#change-data-capture-with-aws-dms)
- [Choosing a database on AWS](#choosing-a-database-on-aws)
- [DynamoDB for data engineers](#dynamodb-for-data-engineers)
- [ElastiCache in data pipelines](#elasticache-in-data-pipelines)
- [Orchestrating pipelines with Step Functions and EventBridge](#orchestrating-pipelines-with-step-functions-and-eventbridge)
- [Least-privilege IAM policies for data pipelines](#least-privilege-iam-policies-for-data-pipelines)
- [Securing data engineering workloads on AWS](#securing-data-engineering-workloads-on-aws)
- [Monitoring data pipelines with CloudWatch](#monitoring-data-pipelines-with-cloudwatch)
- [Controlling cost in AWS data pipelines](#controlling-cost-in-aws-data-pipelines)
- [Choosing between AWS data services: trade-offs](#choosing-between-aws-data-services-trade-offs)
- [Things to remember: key numbers and limits](#things-to-remember-key-numbers-and-limits)
- [Things to remember: semantics to memorize](#things-to-remember-semantics-to-memorize)
- [Things to remember: mistakes that keep coming back](#things-to-remember-mistakes-that-keep-coming-back)
- [Real-world use cases on AWS](#real-world-use-cases-on-aws)
- [AWS exercises](#aws-exercises)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## The AWS mental model for data engineering


Almost every AWS data platform is the same five layers with governance wrapped around them. Learn the layers and each new service slots into a place you already understand.


| Layer | Question it answers | Main AWS services |
|---|---|---|
| Ingest | How does data arrive: batch files, CDC, or streams? | Kinesis Data Streams / Firehose, MSK, DMS, API Gateway |
| Store | Where do the bytes live durably and cheaply? | S3 in raw / curated / analytics zones, with lifecycle rules |
| Catalog & govern | Who can see which tables, columns, and rows? | Glue Data Catalog, Lake Formation, IAM, KMS |
| Process | What transforms raw into curated? | Glue (managed Spark), EMR (your Spark fleet), Lambda/ECS for small jobs |
| Serve | Who queries it, and how fast? | Athena for ad-hoc SQL, Redshift for high-concurrency BI, DynamoDB for point lookups |

Cross-cutting layers sit beside these: orchestration (Step Functions, EventBridge, MWAA), observability (CloudWatch, CloudTrail), and cost control (tags, Budgets, workgroup scan limits).


## S3 for data engineering: objects, consistency, and cost


S3 (Simple Storage Service) stores and retrieves any amount of data via a REST API with pay-as-you-go pricing for storage, requests, and transfer. Since December 2020 S3 provides strong read-after-write consistency for all operations, which is what makes it safe as a lakehouse storage layer.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import boto3

s3 = boto3.client("s3")
s3.put_object(Bucket="acme-raw", Key="orders/ds=2026-10-04/orders.json",
              Body=b'{"order_id": 1001, "amount": 42.50}\n',
              ServerSideEncryption="aws:kms")
print(s3.get_object(Bucket="acme-raw", Key="orders/ds=2026-10-04/orders.json")["Body"].read())

S3 stores objects in buckets. An object has a key, a value, metadata, and a version id, and the slashes in a key are just characters rather than directories. There is no rename operation: a move is a copy plus a delete. Request rate scales per prefix, so spreading writers across prefixes raises aggregate throughput.

Consistency: since December 2020 S3 is strongly consistent for PUT, overwrite, and DELETE, including list operations, in every Region. That removes the old read-after-write workaround of sleeping or retrying a listing.

Common mistake that still costs money: writing one tiny object per event, then paying a request charge per read and listing millions of keys. Batch small records into files of roughly 128 MB to 1 GB.

## S3 storage classes and lifecycle rules


Common classes: S3 Standard (hot data), Intelligent-Tiering (unknown access patterns), Standard-IA and One Zone-IA (infrequent access), Glacier Instant Retrieval, Glacier Flexible Retrieval, and Glacier Deep Archive (archives), plus S3 Express One Zone for latency-sensitive analytics. Use lifecycle rules to transition objects automatically, for example `STANDARD_IA` at 30 days and `GLACIER_IR` at 90 days for a `raw/` prefix.

Watch the IA/Glacier minimum billable duration and per-object overhead: these classes are a poor fit for millions of tiny files.

Every class trades retrieval cost and latency for storage price. Standard has no retrieval fee. Standard-IA and One Zone-IA charge retrieval and have a 30 day minimum billable duration plus a 128 KB minimum billable object size. Glacier Instant Retrieval is milliseconds, Glacier Flexible Retrieval is minutes, and Deep Archive is hours. Intelligent-Tiering moves objects between access tiers automatically for a small per-object monitoring charge.

Lifecycle rules match a prefix or tag and apply `Transition` after a number of days and `Expiration` or `NoncurrentVersionExpiration` for deletion. Transitions into a Glacier class cannot be cheaply reversed: early deletion and retrieval are both billed.

Common mistake: point a lifecycle rule at a prefix holding millions of tiny files. The 128 KB minimum billable size multiplies the effective cost and the per-object transition requests can exceed the storage saved. Compact small files before tiering.

## Multipart upload and large files


Yes, with multipart upload: the object is split into parts uploaded independently and in parallel, then merged. The CLI and SDKs do this automatically above a threshold. Limits: a single PUT is capped at 5 GiB, a part is 5 MiB to 5 GiB, and an object is up to 5 TiB.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import boto3
from boto3.s3.transfer import TransferConfig

config = TransferConfig(multipart_threshold=64 * 1024 * 1024, max_concurrency=16)
boto3.client("s3").upload_file("bigfile.parquet", "acme-raw", "bigfile.parquet", Config=config)

Multipart upload splits an object into parts, uploads them in parallel, and completes them into one object. Limits: a single PUT is capped at 5 GiB, each part is 5 MiB to 5 GiB except the last, and the completed object can be up to 5 TiB. Incomplete uploads stay in the bucket and are billed until aborted.

Common mistake: leaving abandoned multipart uploads behind after a failed job. Add a bucket lifecycle rule with `AbortIncompleteMultipartUpload` after a few days, or a crashing nightly load quietly accumulates terabytes of billable orphan parts.

## Designing an AWS data lake: raw, curated, analytics


```text
s3://acme-lake/raw/orders/ds=2026-10-04/          # immutable, as landed
s3://acme-lake/curated/orders/ds=2026-10-04/      # typed, deduplicated Parquet
s3://acme-lake/analytics/fact_orders/ds=2026-10-04/
```

The raw, curated, and analytics zones are conventions rather than services. S3 holds the bytes, the Glue Data Catalog holds table metadata, Lake Formation holds permissions, and Glue, EMR, Athena, and Redshift do the compute on top.

Governance matters as much as storage: bucket policies, IAM roles, KMS encryption, Lake Formation permissions, lifecycle rules, and CloudTrail. A lake needs metadata, access control, data quality, and cost management - not just files.

```yaml
AWSTemplateFormatVersion: "2010-09-09"
Resources:
  RawBucket:
    Type: AWS::S3::Bucket
    Properties:
      BucketName: acme-lake-raw
      VersioningConfiguration:
        Status: Enabled
      BucketEncryption:
        ServerSideEncryptionConfiguration:
          - ServerSideEncryptionByDefault:
              SSEAlgorithm: aws:kms
              KMSMasterKeyID: alias/acme-lake
      LifecycleConfiguration:
        Rules:
          - Id: abort-multipart
            Status: Enabled
            AbortIncompleteMultipartUpload:
              DaysAfterInitiation: 3
          - Id: tier-logs
            Status: Enabled
            Prefix: logs/
            Transitions:
              - StorageClass: STANDARD_IA
                TransitionInDays: 30
```

Common mistake: making the raw zone mutable. If raw is overwritten in place, a transformation bug cannot be replayed; keep raw immutable and versioned and rebuild curated from it.

## Tuning S3 layout and file sizes for Athena and Glue


S3 is a flat namespace of immutable objects. The slashes in a key are characters, a listing is served per prefix, and every GET, PUT, and LIST is billed as a request. That shapes lake layout more than any engine setting: Spark reads one task per file, so a prefix with 500,000 four-megabyte files creates 500,000 tasks, 500,000 GET requests, and a planning cost that dwarfs the actual scan. Target files of roughly 128 MB to 1 GB for columnar data.

Convert to Parquet or ORC with compression (Snappy is the usual default, ZSTD compresses smaller) so a query reads only the columns it references and only the row groups its predicates match. Then partition by the columns that appear in filters, ordered from the most selective low-cardinality column outward, and use Hive-style keys such as `ds=2026-10-04/`. Partitioning too finely is its own problem: tens of thousands of partitions slow planning and produce tiny files, which is why partition projection plus coarser day or hour granularity is usually the right balance.

```sql
-- Compact a small-file prefix into one file per partition, one time.
CREATE TABLE curated.orders_compact
WITH (
  format = 'PARQUET',
  parquet_compression = 'ZSTD',
  partitioned_by = ARRAY['ds'],
  bucketed_by = ARRAY['customer_id'],
  bucket_count = 16
) AS
SELECT order_id, customer_id, order_ts, amount, ds
FROM curated.orders
WHERE ds BETWEEN '2026-09-01' AND '2026-09-30';
```

```bash
# See the real file-size distribution before you tune anything.
aws s3api list-objects-v2 --bucket acme-curated --prefix orders/ \
  --query 'Contents[].{Key:Key,MB:Size}' --output text \
  | awk '{sum+=$2; n++; if ($2 < 8388608) small++} END {printf "files=%d avg=%.1fMB small=%d\n", n, sum/n/1048576, small}'
```

Common mistake: partitioning by a high-cardinality column such as `customer_id`. That creates one prefix per customer, each holding a few tiny files, so both listing and planning explode. Keep the date partition and bucket or cluster by the high-cardinality column inside it instead.

## Governing the lake with AWS Lake Formation


Lake Formation centralizes fine-grained permissions on Glue Data Catalog tables and the S3 data behind them. You register S3 locations, grant table/column/row permissions to principals, and scale with LF-Tags instead of per-table grants.

```bash
aws lakeformation register-resource --resource-arn arn:aws:s3:::acme-curated --use-service-linked-role
aws lakeformation add-lf-tags --catalog-id 123456789012 --tag-key classification --tag-values public pii phi
aws lakeformation grant-permissions \
  --principal DataLakePrincipalIdentifier=arn:aws:iam::123456789012:role/analyst-role \
  --resource '{"TableWithColumns":{"DatabaseName":"curated","Name":"orders","ColumnNames":["order_id","order_ts","amount"],"ColumnWildcard":{}}}' \
  --permissions SELECT
```

Lake Formation manages grants on catalog resources and enforces them by vending temporary credentials and filtering rows and columns. `SELECT` can be narrowed with a data filter, and LF-Tags attach key and value labels to resources and principals so you grant by tag instead of table by table.

It provides column-level and row-level security, tag-based access, and cross-account sharing, but it does not replace IAM and only governs engines that honor it (Athena, Glue, Redshift Spectrum, and EMR with the Lake Formation runtime).

Common mistake: registering an S3 location with Lake Formation but leaving a bucket policy that still grants direct `s3:GetObject`. Any principal with that IAM permission bypasses Lake Formation enforcement completely, so access must go through a governed engine or an explicit direct grant.

## Glue, EMR, Lambda, or ECS: picking the compute for a job


Use Glue for managed, catalog-integrated Spark/Python ETL at moderate scale. Use EMR for large, shuffle-heavy, or custom Spark/Hadoop workloads and Spot-heavy batch. Use Lambda for small event-driven tasks under 15 minutes. Use ECS/Fargate for containerized jobs that need more runtime control than Lambda but no Spark. The choice is driven by data size, runtime, dependency complexity, and how much cluster operations you want to own.

Glue is managed Spark with a scheduler, catalog integration, and worker billing by the second; it auto-scales executors within the minimum and maximum worker range you set. EMR gives you the cluster itself: instance fleets, a Spot mix, bootstrap actions, and direct Spark configuration. Lambda is limited to 15 minutes, 10 GB of memory, and 512 MB of writable `/tmp`.

Glue sizing mechanics: a job runs on G.1X (4 vCPU, 16 GB), G.2X (8 vCPU, 32 GB), G.4X, or G.8X workers, and `NumberOfWorkers` sets the initial fleet. Older jobs were billed in DPUs, where one DPU is 4 vCPU and 16 GB, which is why DPU-hour and worker-hour numbers appear side by side in older material.

Common mistake: choosing EMR for a job that runs ten minutes a day. Cluster startup and bootstrap dominate the cost while Glue or a scheduled Lambda is cheaper and has less to operate. The reverse mistake is running a multi-hour shuffle-heavy job on Glue when an EMR fleet with Spot task nodes costs less.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import boto3

glue = boto3.client("glue")

run = glue.start_job_run(
    JobName="orders_curated",
    Arguments={
        "--raw_path": "s3://acme-raw/orders/",
        "--curated_path": "s3://acme-curated/orders/",
    },
    WorkerType="G.1X",
    NumberOfWorkers=20,
    Timeout=60,
)
print(run["JobRunId"])


## Writing an AWS Glue PySpark job


A Glue job is managed Spark. Read job parameters with `getResolvedOptions`, use `GlueContext` for catalog-aware IO, transform with Spark DataFrames, and write partitioned Parquet; call `job.commit()` to advance job bookmarks.

In [ ]:
# ILLUSTRATIVE - requires awsglue, pyspark: install to run, or read on.
import sys

from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as F

args = getResolvedOptions(sys.argv, ["JOB_NAME", "raw_path", "curated_path"])
sc = SparkContext()
glue_context = GlueContext(sc)
spark = glue_context.spark_session
job = Job(glue_context)
job.init(args["JOB_NAME"], args)

curated = (
    spark.read.json(args["raw_path"])
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("ds", F.to_date("order_ts"))
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
)
curated.write.mode("overwrite").partitionBy("ds").parquet(args["curated_path"])
job.commit()

Operational notes: enable job metrics and continuous logging, set worker type/size and a timeout, and consider `--enable-auto-scaling`. For very large shuffle-heavy jobs, EMR with Spot task nodes is often cheaper.

A Glue job is a Spark application with a managed driver and executor fleet, a Glue Data Catalog client, and job bookmarks. `getResolvedOptions` reads job parameters, `GlueContext` adds catalog and DynamicFrame IO, and `job.commit()` persists bookmark state.

Job bookmarks track the input already processed, but the job must call `commit()` and the transform must be deterministic. Worker sizing is per job: for a 500 GB Parquet input with a moderate shuffle, start around 10 G.1X workers and watch the shuffle spill metrics rather than guessing. Common mistake: enabling bookmarks on a job that writes with `mode("overwrite")` and then wondering why a retry reprocesses the same files, or leaving auto scaling off so the job holds a fixed worker count for the whole run.

```sql
-- What does the catalog actually know about the curated database?
SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'curated'
ORDER BY table_name, ordinal_position;

-- Check which partitions a job has registered, without reading data.
SHOW PARTITIONS curated.orders;
```

## Tuning Glue job sizing and bookmarks


A Glue job is Spark on a managed fleet. Worker types are G.1X (4 vCPU, 16 GB), G.2X (8 vCPU, 32 GB), G.4X, and G.8X, and one worker equals one DPU of the older billing model. `NumberOfWorkers` sets the initial fleet, `--enable-auto-scaling` lets the job add and remove executors within the minimum and maximum, and `--worker-type` and `--number-of-workers` are real job parameters. Billing is per DPU-hour, rounded by the second with a one-minute minimum, which is why a job that idles for an hour waiting on a lock is expensive.

Job bookmarks record which input was already processed so a scheduled rerun reads only new files. They only work when the job calls `job.commit()`, when the input is read through the bookmarked source, and when the transform is deterministic. `--job-bookmark-option` takes `job-bookmark-enable`, `job-bookmark-disable`, and `job-bookmark-pause`. A bookmark tracks input state, not output correctness, so a job that rewrites a partition must be idempotent on its own.

```bash
aws glue create-job --name orders_curated \
  --role arn:aws:iam::123456789012:role/glue-orders-role \
  --worker-type G.1X --number-of-workers 20 \
  --default-arguments '{
    "--job-bookmark-option": "job-bookmark-enable",
    "--enable-auto-scaling": "true",
    "--enable-metrics": "true",
    "--enable-continuous-cloudwatch-log": "true",
    "--conf": "spark.sql.shuffle.partitions=400",
    "--TempDir": "s3://acme-glue-temp/orders/"
  }' \
  --command '{"Name":"glueetl","ScriptLocation":"s3://acme-scripts/orders_curated.py","PythonVersion":"3"}'
```

```json
{
  "JobName": "orders_curated",
  "WorkerType": "G.2X",
  "NumberOfWorkers": 10,
  "Timeout": 60,
  "GlueVersion": "4.0",
  "ExecutionProperty": {"MaxConcurrentRuns": 1}
}
```

Common mistake: leaving the default shuffle partition count on a small job, which creates hundreds of near-empty tasks, or setting it far too low on a large aggregation, which creates a few enormous partitions that spill. Size shuffle partitions from the shuffle volume, and treat a single hot key as skew to be salted or pre-aggregated rather than as a reason to add workers.

## EMR fleets and Spot capacity for batch processing


EMR cost is a fleet decision. An instance fleet can mix a small On-Demand core group with a large Spot task group, and the allocation strategy decides how Spot capacity is spread across instance types and Availability Zones. Bootstrap actions run before the application starts and are the place for extra libraries or Spark defaults; on EMR the default filesystem is EMRFS over S3, so always use the S3 committer rather than writing through a local path. Core nodes hold HDFS data, so removing them requires careful decommissioning; task nodes hold no HDFS data and are the safe Spot group.

```json
{
  "InstanceFleets": [
    {"Name": "primary", "InstanceFleetType": "MASTER",
     "TargetOnDemandCapacity": 1,
     "InstanceTypeConfigs": [{"InstanceType": "r6i.2xlarge"}]},
    {"Name": "core", "InstanceFleetType": "CORE",
     "TargetOnDemandCapacity": 8,
     "InstanceTypeConfigs": [{"InstanceType": "r6i.2xlarge"}, {"InstanceType": "r6a.2xlarge"}]},
    {"Name": "task", "InstanceFleetType": "TASK",
     "TargetSpotCapacity": 16,
     "LaunchSpecifications": {"SpotSpecification": {"TimeoutDurationMinutes": 20,
       "TimeoutAction": "SWITCH_TO_ON_DEMAND", "AllocationStrategy": "capacity-optimized"}},
     "InstanceTypeConfigs": [{"InstanceType": "r6i.2xlarge"}, {"InstanceType": "r6a.2xlarge"}, {"InstanceType": "r5.2xlarge"}]}
  ]
}
```

## Querying S3 with Athena and partition projection


Athena is serverless SQL over S3 and the Glue Data Catalog. Partition projection computes partition locations from the table definition, so you avoid `MSCK REPAIR TABLE` and crawler cost across tens of thousands of partitions.

```sql
CREATE EXTERNAL TABLE curated.orders (
  order_id bigint, customer_id bigint, order_ts timestamp, amount decimal(12,2)
)
PARTITIONED BY (ds string) STORED AS PARQUET
LOCATION 's3://acme-curated/orders/'
TBLPROPERTIES (
  'projection.enabled' = 'true',
  'projection.ds.type' = 'date',
  'projection.ds.range' = '2024-01-01,NOW',
  'projection.ds.format' = 'yyyy-MM-dd',
  'projection.ds.interval' = '1',
  'projection.ds.interval.unit' = 'DAYS',
  'storage.location.template' = 's3://acme-curated/orders/ds=${ds}/'
);

SELECT ds, count(*) AS orders, sum(amount) AS revenue
FROM curated.orders
WHERE ds BETWEEN '2026-09-01' AND '2026-10-04'
GROUP BY ds ORDER BY ds;
```

Athena bills by bytes scanned, so partition pruning, Parquet/ORC, compression, and selecting only needed columns matter more than engine size. Avoid `SELECT *` and wide raw-JSON scans, and set a per-query bytes-scanned cutoff in the workgroup.

Athena engine v3 runs Trino and engine v2 runs Presto. Every query runs inside a workgroup that sets the output location, encryption, a per-query bytes-scanned cutoff, and whether settings are enforced. `CTAS` writes a new table from a query and `INSERT INTO` appends, optionally writing new partitions.

Partition projection turns a partition filter into a location pattern computed from the table properties, so no catalog partitions are looked up and `MSCK REPAIR TABLE` is unnecessary. The current property is `partition_projection.enabled`; older examples use `projection.enabled`, which still works but is the legacy name.

Common mistake: leaving the table format as CSV or JSON because the files already have headers. Athena bills by compressed bytes read, so Parquet over the same data often scans an order of magnitude less and can skip columns the query never references.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import time

import boto3

athena = boto3.client("athena")


def run_query(sql: str, database: str, workgroup: str) -> str:
    execution_id = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
        WorkGroup=workgroup,
    )["QueryExecutionId"]
    while True:
        status = athena.get_query_execution(QueryExecutionId=execution_id)["QueryExecution"]["Status"]
        if status["State"] in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            print(status["State"], status.get("StateChangeReason", ""))
            return execution_id
        time.sleep(1)


run_query("SELECT count(*) FROM curated.orders WHERE ds = '2026-10-04'", "curated", "analytics-guarded")


## Tuning Athena cost and speed


Athena bills by bytes scanned at roughly 5 USD per TB in us-east-1, with a 10 MB minimum per query. DDL statements such as `CREATE TABLE` are not billed, and a query that fails before it reads data is not billed. Everything that reduces scanned bytes reduces cost: columnar format, compression, partition pruning, and selecting only the columns the answer needs. Engine v3 runs Trino and is the default for new workgroups; engine v2 runs Presto.

A workgroup is the control point. It sets the query result location, enforces encryption, can require a client-side setting, and can cap `bytes scanned per query` so a runaway statement fails instead of billing. Partition projection computes partition locations from table properties so the catalog does not have to hold, and the engine does not have to plan, every partition. `CTAS` writes the result of a query into a new Parquet table, which is how you turn a repeated expensive scan into a cheap one.

```sql
-- Projection from a table definition, not from the catalog.
ALTER TABLE curated.events SET TBLPROPERTIES (
  'partition_projection.enabled' = 'true',
  'partition_projection.ds.type' = 'date',
  'partition_projection.ds.range' = '2024-01-01,NOW',
  'partition_projection.ds.format' = 'yyyy-MM-dd',
  'partition_projection.ds.interval' = '1',
  'partition_projection.ds.interval.unit' = 'DAYS',
  'storage.location.template' = 's3://acme-curated/events/ds=${ds}/'
);

-- One monthly aggregate scan instead of a dashboard scanning raw JSON forever.
CREATE TABLE analytics.events_daily
WITH (format = 'PARQUET', partitioned_by = ARRAY['ds']) AS
SELECT ds, event_type, count(*) AS events, approx_distinct(user_id) AS users
FROM curated.events
WHERE ds BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY ds, event_type;
```

```bash
# Create a workgroup that refuses any query scanning more than 10 GB.
aws athena create-work-group --name analytics-guarded \
  --configuration 'ResultConfiguration={OutputLocation=s3://acme-athena-results/},EnforceWorkGroupConfiguration=true,BytesScannedCutoffPerQuery=10737418240,PublishCloudWatchMetricsEnabled=true'
```

Common mistake: rebuilding the same aggregation every five minutes directly from raw JSON. Schedule one `CTAS` or `INSERT INTO` into a Parquet aggregate table and let the dashboard read that; the dashboard then scans megabytes instead of terabytes, and the cost becomes predictable.

## Amazon Redshift: architecture and when to use it


Redshift is a fast, scalable, cost-effective cloud data warehouse for gigabytes to hundreds of petabytes. It is popular because it is easy to create and operate, scales by resizing or adding nodes, uses columnar storage and massively parallel processing for fast analytics, costs a fraction of traditional warehouses, and integrates with S3, the Glue Data Catalog, and Spectrum.

Redshift uses columnar storage with per-column compression chosen by `ANALYZE COMPRESSION`, and executes a query in parallel across slices. RA3 separates compute from managed storage, Spectrum offloads S3 scans to an external fleet, and concurrency scaling adds transient read clusters.

A warehouse stores data columnar and compressed, so a query that reads one column does not pay for the rest. Redshift then splits each table across slices on compute nodes; a distribution key aligns matching rows with the join so they land on the same slice.

The leader node receives client queries, parses them, builds the execution plan, coordinates steps, and returns results. Compute nodes execute the assigned steps and return data to the leader. On RA3 clusters, compute is separate from managed storage and concurrency scaling adds transient clusters for read bursts.

The leader node parses SQL, builds the plan, coordinates slices, and merges results; on RA3 it holds no user tables. Compute nodes store the data and run the compiled steps. A leader under load shows up as queue time rather than slow execution.

Common mistake: blaming compute for a slow query when the plan is serialized on the leader. Check the queue and concurrency metrics before resizing the cluster.

Common mistake: loading uncompressed row-oriented data and expecting the warehouse to fix it at query time. `COPY` from Parquet or compressed CSV is the difference between a fast load and an expensive one.

Common mistake: designing the model before the queries. Distribution and sort keys follow from the dominant joins and time filters, so choosing them from an entity diagram alone produces redistribution or broadcast on every join.

## Redshift Spectrum for querying the lake


Spectrum runs SQL against exabytes in S3 without loading it, using external tables usually defined in the Glue Data Catalog and scaling across many nodes for parallel scan and filter.

```sql
CREATE EXTERNAL SCHEMA spectrum_raw FROM DATA CATALOG DATABASE 'raw'
IAM_ROLE 'arn:aws:iam::123456789012:role/redshift-spectrum';
SELECT event_type, count(*) FROM spectrum_raw.events
WHERE event_ts >= '2026-10-01' GROUP BY 1 ORDER BY 2 DESC;
```

Spectrum submits a query plan to a separate fleet of servers that read S3 directly, using external tables defined in the Glue Data Catalog or an external schema. You are billed for the bytes that fleet scans, not for your cluster CPU.

Common mistake: pointing Spectrum at JSON or uncompressed CSV. Columnar Parquet with partitions avoids columns and rows the query does not need, and Spectrum charges per byte scanned. File sizes of 100 MB or more also keep the per-file overhead small.

## Loading and optimizing data in Redshift


Stage files in S3 and use `COPY` rather than inserts, prefer Parquet, compressed and sized so slices work in parallel. Design distribution and sort keys around common joins and time filters, and use `EXPLAIN` and system tables to find skew and full scans.

```sql
COPY fact_orders FROM 's3://acme-curated/orders/'
IAM_ROLE 'arn:aws:iam::123456789012:role/redshift-load-role'
FORMAT AS PARQUET STATUPDATE ON;
ALTER TABLE fact_orders ALTER DISTSTYLE KEY DISTKEY (customer_id);
ALTER TABLE fact_orders ALTER SORTKEY (order_ts);
```

RA3 managed storage, automatic table optimization, materialized views, and concurrency scaling help, but bad file layout and inefficient joins still dominate cost.

A sort key puts rows touched by a range filter into the same blocks so zone maps skip the rest. A compound sort key helps a leading column prefix, while an interleaved sort key gives several columns equal weight. Distribution (`KEY`, `EVEN`, `ALL`, `AUTO`) decides which slice holds each row and therefore which joins are local.

Common mistake: distributing a small dimension `EVEN` and joining it to a large fact, which forces a broadcast or redistribution on every query. Small dimensions belong on `ALL`; large facts distribute on the common join key.

## COPY and UNLOAD: Redshift bulk paths


`COPY` is the parallel load path and `UNLOAD` the parallel export path; both use an IAM role and can write Parquet. Avoid single-file loads because they serialize on one slice.

```sql
COPY fact_orders FROM 's3://acme-curated/orders/'
IAM_ROLE 'arn:aws:iam::123456789012:role/redshift-load-role'
FORMAT AS PARQUET STATUPDATE ON;

UNLOAD ('SELECT * FROM fact_orders WHERE order_ts >= ''2026-10-01''')
TO 's3://acme-analytics/exports/fact_orders/'
IAM_ROLE 'arn:aws:iam::123456789012:role/redshift-unload-role'
FORMAT AS PARQUET PARTITION BY (ds) ALLOWOVERWRITE PARALLEL ON;
```

Use a manifest for a controlled file list, stage via S3 even from databases (UNLOAD or CDC), and tune `COMPUPDATE`/`STATUPDATE`.

`COPY` distributes input files across slices and `UNLOAD` writes one or more files per slice into S3 with an optional manifest. Both accept an IAM role, and the role form is the one to use rather than embedded credentials.

Common mistake: unloading to a single file with `PARALLEL OFF`. That serializes the export and produces one huge object that a later `COPY` cannot parallelize.

## Kinesis, MSK, SQS, and EventBridge: how they differ


Kinesis Data Streams is ordered, shard-based streaming with replay within retention. MSK is managed Kafka for teams already on Kafka APIs and tooling. SQS is a decoupling queue for message-at-a-time work, not analytics streams. EventBridge routes events and schedules. Compare ordering, replay, retention, throughput, consumer model, and operational cost: MSK is more powerful but you own brokers; Kinesis is simpler but quotas shape the design.

Kinesis Data Streams assigns each record to a shard by the hash of its partition key, and each consumer tracks a sequence number per shard. MSK gives Kafka topics, partitions, and consumer groups, but you own broker sizing and upgrades. SQS is a queue with a visibility timeout and no global ordering, unless you use a FIFO queue. EventBridge is a router and scheduler, not a buffer.

Common mistake: using SQS for an analytics stream and then discovering you need replay. Messages are deleted on acknowledgement, while Kinesis and MSK retain records for the retention period so a consumer can rewind.

## Ingesting streams with Amazon Kinesis


Kinesis Data Streams is the replayable low-level stream and Firehose delivers to S3, Redshift, or OpenSearch with optional transformation. A producer partitions records by key; consumers read shards with an iterator.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import json

import boto3

kinesis = boto3.client("kinesis")


def publish(stream_name: str, orders: list[dict]) -> None:
    records = [
        {"PartitionKey": str(order["customer_id"]), "Data": (json.dumps(order) + "\n").encode("utf-8")}
        for order in orders
    ]
    kinesis.put_records(StreamName=stream_name, Records=records)

Design points: the partition key controls ordering and hot-shard risk; one shard supports 1 MB/s or 1,000 records/s ingest and 2 MB/s egress; retention defaults to 24 hours and can extend to 365 days for replay; use enhanced fan-out for multiple fast consumers and on-demand mode to avoid shard math. Firehose buffers up to 900 seconds, so it is near-real-time.

A shard accepts 1 MB/s or 1,000 records/s of ingest, whichever limit is hit first, and provides 2 MB/s of classic read egress shared by all consumers. The partition key hash picks the shard, so a high-cardinality key spreads load. Enhanced fan-out gives each registered consumer its own 2 MB/s per shard over HTTP/2 so one slow consumer cannot consume the shared read budget.

Firehose buffers by size (1 to 128 MB) and interval (60 to 900 seconds) and delivers to S3, Redshift, or OpenSearch with optional Lambda transformation, retries, and JSON-to-Parquet conversion. On-demand stream mode scales shards automatically with a default 200 MB/s write ceiling.

Common mistake: using a low-cardinality partition key such as status or a single tenant id. One shard then receives every record for that key and throttles with `ProvisionedThroughputExceededException` while the other shards sit idle.

Kinesis Data Streams charges per shard-hour plus per PUT payload unit. One shard takes 1 MB/s or 1,000 records/s of ingest, whichever comes first, and gives 2 MB/s of shared read. Retention defaults to 24 hours and can be extended to 365 days, which adds a per-GB retention charge. Provisioned mode asks you to do the shard math; on-demand mode scales automatically and is the simpler default for spiky traffic. Enhanced fan-out gives a dedicated 2 MB/s per shard per registered consumer and is billed separately.

Common mistake: sizing shards for the promotional peak and paying for them all year. Provision for the normal load and let on-demand mode or a resharding step absorb the spike, and check whether Firehose (billed per GB delivered) is a better fit than a stream you have to size at all.

## Change data capture with AWS DMS


AWS DMS (Database Migration Service) moves data from source databases to AWS targets in two modes: a one-time **full load**, and **CDC (change data capture)** that reads inserts, updates, and deletes from the source's transaction log and applies them continuously. Sources include RDS/Aurora, PostgreSQL, MySQL, Oracle, and SQL Server; targets include S3, Redshift, and Kinesis or MSK for streaming consumers.


For a lake, an S3 target writes change files per table with operation metadata, partitioned however you configure it; for near-real-time pipelines, a Kinesis or MSK target feeds a streaming Glue job that dedupes by primary key plus log sequence number. The replication instance is the compute that does the work — size it to the change rate, not the table size.


In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import json

import boto3

dms = boto3.client("dms")

task = dms.create_replication_task(
    ReplicationTaskIdentifier="orders-cdc-to-s3",
    SourceEndpointArn="arn:aws:dms:us-east-1:123456789012:endpoint:orders-src",
    TargetEndpointArn="arn:aws:dms:us-east-1:123456789012:endpoint:lake-raw",
    ReplicationInstanceArn="arn:aws:dms:us-east-1:123456789012:rep:cdc-instance",
    MigrationType="full-load-and-cdc",
    TableMappings=json.dumps({
        "rules": [{
            "rule-type": "selection",
            "rule-id": "1",
            "rule-name": "orders",
            "object-locator": {"schema-name": "public", "table-name": "orders"},
            "rule-action": "include",
        }]
    }),
)
print(task["ReplicationTask"]["ReplicationTaskArn"])


Common mistake: treating DMS as a daily export tool and re-running full loads. A repeated full load duplicates every row downstream and scales with table size; the steady state should be `full-load-and-cdc`, where the full load bootstraps once and CDC keeps the target tracking the source log.


## Choosing a database on AWS


AWS offers managed relational (RDS, Aurora), NoSQL key-value/document (DynamoDB), warehouse (Redshift), and in-memory (ElastiCache, MemoryDB) databases. Choose by access pattern, scale, consistency, and query flexibility - not by brand loyalty.

Pick by access pattern rather than brand: relational (RDS, Aurora) for transactions and joins, key-value (DynamoDB) for high-volume lookups, warehouse (Redshift) for analytics, in-memory (ElastiCache, MemoryDB) for microsecond reads, and specialized engines for graph or time series.

Common mistake: using one database for every phase of a pipeline. The OLTP database, the lake, the warehouse, and the serving cache each have a different access pattern, and forcing one engine to serve all four produces both slow queries and a large bill.

## DynamoDB for data engineers


DynamoDB is the fully managed, serverless key-value/document store with consistent single-digit millisecond latency at any scale. Data engineers meet it in two roles: as an OLTP **source** a pipeline extracts from, and as a **serving layer** for pre-computed features and aggregates that need point lookups, not scans.


Getting data out for analytics: DynamoDB Streams captures every write for CDC into Kinesis/Firehose and on to the lake, and point-in-time **export to S3** produces a full snapshot in DynamoDB JSON (Athena-queryable after conversion) without consuming table read capacity. Modeling rules to respect: high-cardinality partition keys spread load, a GSI reshapes data for a second access pattern, items are capped at 400 KB so large payloads belong in S3 with a pointer in the item, and a `Scan` with `FilterExpression` pays to read every item before filtering.


Common mistake: running analytics as table scans against production DynamoDB. The scan competes with live traffic for capacity and bills for every item read; export to S3 or stream changes to the lake and query that copy with Athena instead.


## ElastiCache in data pipelines


ElastiCache is managed Valkey/Redis or Memcached: sub-millisecond in-memory lookups. In pipelines it shows up as an enrichment lookup cache or as a read-through buffer in front of a database that a hot job would otherwise hammer. It is never a source of truth — set a TTL and define how the cache is rebuilt after it is lost.


Common mistake: caching without an invalidation story. Without a TTL or publish-on-write invalidation, enrichment jobs serve stale values silently, and the staleness is invisible until a downstream number is questioned.


## Orchestrating pipelines with Step Functions and EventBridge


Step Functions coordinate multi-step workflows with retries, catch blocks, and parallel branches; EventBridge triggers them on schedules or events such as an S3 `ObjectCreated`. Using `.sync` integration patterns makes the state machine wait for Glue and Athena.

```json
{
  "Comment": "Order file arrival to Glue to Athena validation",
  "StartAt": "RunGlue",
  "States": {
    "RunGlue": {
      "Type": "Task",
      "Resource": "arn:aws:states:::glue:startJobRun.sync",
      "Parameters": {"JobName": "orders_curated"},
      "Retry": [{"ErrorEquals": ["States.TaskFailed"], "IntervalSeconds": 30, "MaxAttempts": 3, "BackoffRate": 2.0}],
      "Catch": [{"ErrorEquals": ["States.ALL"], "Next": "NotifyFailure"}],
      "Next": "Validate"
    },
    "Validate": {
      "Type": "Task",
      "Resource": "arn:aws:states:::athena:startQueryExecution.sync",
      "Parameters": {"QueryString": "SELECT count(*) FROM curated.orders WHERE ds = current_date", "WorkGroup": "primary"},
      "Next": "Success"
    },
    "Success": {"Type": "Succeed"},
    "NotifyFailure": {
      "Type": "Task",
      "Resource": "arn:aws:states:::sns:publish",
      "Parameters": {"TopicArn": "arn:aws:sns:us-east-1:123456789012:data-oncall", "Message": "orders pipeline failed"},
      "End": true
    }
  }
}
```

A state machine is JSON built from `Task`, `Choice`, `Parallel`, `Map`, and `Wait` states. `arn:aws:states:::glue:startJobRun.sync` starts a Glue job and waits for completion, and `Retry` and `Catch` handle failures per state. Express workflows suit high-volume short runs, while Standard workflows support long durations and exactly-once semantics.

```json
{"Source": ["aws.s3"], "DetailType": ["Object Created"],
 "Detail": {"bucket": {"name": ["acme-raw"]}, "object": {"key": [{"prefix": "orders/"}]}}}
```

EventBridge rules match an event pattern and route it to targets, and a scheduled rule uses a cron or rate expression. EventBridge retries a target according to its retry policy and then sends failures to a dead-letter queue; it is a router, not a durable data buffer.

Interview points: Step Functions provide retry/backoff and idempotency hooks cron jobs lack, and EventBridge replaces polling with event-driven and scheduled triggers.

Common mistake: writing orchestration that is not idempotent. Step Functions retry a failed state, so the Glue job it starts must tolerate a re-run; rely on bookmarks, overwriting one partition, or a dedupe key rather than assuming a single execution.

For teams standardized on Airflow, MWAA (Managed Workflows for Apache Airflow) runs Airflow on AWS and schedules Glue, EMR, and Redshift work as DAGs instead of state machines. The same idempotency rule applies: any retried task must be safe to re-run, so write to one partition at a time or gate on a dedupe key.


## Least-privilege IAM policies for data pipelines


Build one role per pipeline, grant only the needed actions on specific ARNs, add encryption and network conditions, and use permissions boundaries. Prefer KMS key policies plus IAM to enforce encryption.

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {"Sid": "ReadRaw", "Effect": "Allow", "Action": ["s3:GetObject"],
     "Resource": "arn:aws:s3:::acme-raw/orders/*"},
    {"Sid": "WriteCurated", "Effect": "Allow", "Action": ["s3:PutObject", "s3:AbortMultipartUpload"],
     "Resource": "arn:aws:s3:::acme-curated/orders/*",
     "Condition": {"StringEquals": {"s3:x-amz-server-side-encryption": "aws:kms"}}},
    {"Sid": "Catalog", "Effect": "Allow",
     "Action": ["glue:GetTable", "glue:GetDatabase", "glue:GetPartitions", "glue:UpdateTable"],
     "Resource": ["arn:aws:glue:us-east-1:123456789012:catalog",
                  "arn:aws:glue:us-east-1:123456789012:database/curated",
                  "arn:aws:glue:us-east-1:123456789012:table/curated/orders"]},
    {"Sid": "Kms", "Effect": "Allow", "Action": ["kms:Decrypt", "kms:GenerateDataKey"],
     "Resource": "arn:aws:kms:us-east-1:123456789012:key/abcd-1234"}
  ]
}
```

Anti-patterns to call out: `"Resource": "*"` with `s3:*`, wildcard `iam:PassRole`, credentials in job configs, and one shared role across unrelated pipelines.

A policy is JSON with `Effect`, `Action`, `Resource`, and optional `Condition`. Object actions need the object ARN while `s3:ListBucket` needs the bucket ARN, so they are separate statements. `iam:PassRole` with a wildcard resource lets a principal hand any role to a service, which is a known privilege-escalation path.

Common mistake: granting `s3:GetObject` on `bucket/*` for a job that reads one prefix, then adding `s3:*` to be safe after a failure. Read the `AccessDenied` error for the exact action and resource and add only that.

## Securing data engineering workloads on AWS


Use least-privilege IAM roles instead of access keys, KMS encryption at rest and TLS in transit, private subnets, security groups, and VPC endpoints, plus CloudTrail and service logs for audit. Block public S3 access. Build security into the design with one role per job, separate raw and curated permissions, no shared admin roles, and ownership tags.

Four layers: network (private subnets, security groups, VPC endpoints), identity (one role per job, no access keys, permissions boundaries), data (KMS CMKs, TLS, bucket policies that deny unencrypted puts), and audit (CloudTrail, S3 access logs, Lake Formation grants).

Common mistake: encrypting with the default `aws/s3` key and then being unable to scope who can decrypt. Use a customer-managed KMS key per data domain and add a bucket policy condition that denies `s3:PutObject` without the matching encryption header.

## Monitoring data pipelines with CloudWatch


CloudWatch monitors AWS resources, stores logs and custom metrics, and triggers alarms on thresholds. For pipelines the useful pieces are one log group per job, metric filters for row counts or failures, and alarms routed to SNS or a paging tool.

```bash
aws cloudwatch put-metric-alarm --alarm-name glue-orders-failed \
  --metric-name glue.driver.aggregate.numFailedTasks --namespace Glue \
  --statistic Sum --period 300 --threshold 1 \
  --comparison-operator GreaterThanOrEqualToThreshold --evaluation-periods 1 \
  --alarm-actions arn:aws:sns:us-east-1:123456789012:data-oncall \
  --dimensions Name=JobName,Value=orders_curated
```

Metrics are namespaced time series with dimensions, logs live in log groups and streams, and alarms evaluate one metric over a period and an evaluation window. Glue, EMR, and Kinesis publish their own namespaces, and you pay separately for custom metrics, log ingestion, and log storage.

An alarm needs a namespace, metric name, dimensions, statistic, period, and evaluation periods. With a 300 second period and three evaluation periods, the alarm needs three consecutive bad datapoints before it fires. Gotcha: alarming on a raw count without a long enough period turns normal batch variance into constant paging.

## Controlling cost in AWS data pipelines


Reduce scanned data, choose the right compute, use lifecycle policies, right-size jobs, and shut down idle resources. Convert CSV/JSON to compressed Parquet, partition by common filters, expire staging data with S3 lifecycle rules, use Spot for fault-tolerant EMR, enable managed/auto scaling, and add cost allocation tags with Budgets alarms. Inefficient layout and unbounded scans usually cost more than an expensive instance.

The cost model is per unit of work: Athena and Spectrum per byte scanned, Glue and EMR per worker-hour or DPU-hour, Redshift per cluster-hour plus concurrency scaling, Kinesis per shard-hour plus PUT payload units, MSK per broker-hour, S3 per GB-month plus requests and retrieval, and data transfer per GB across Availability Zones, Regions, and to the internet.

Apply the cheap fixes first: fewer bytes (partition, columnar format, compression), fewer idle hours (auto-stop, TTL, lifecycle rules), and cheaper capacity (Spot, Savings Plans, Graviton). Cost allocation tags plus Cost Explorer and Budgets make the spend attributable, which is what makes it manageable.

Common mistake: optimizing the instance type before the data layout. A tenfold reduction in bytes scanned cuts Athena spend tenfold with no infrastructure change, while a 20 percent cheaper instance is noise by comparison.

```bash
# Abort abandoned multipart uploads and tier cold prefixes automatically.
aws s3api put-bucket-lifecycle-configuration --bucket acme-lake \
  --lifecycle-configuration '{"Rules":[
    {"ID":"abort-mpu","Status":"Enabled","Filter":{"Prefix":"raw/"},"AbortIncompleteMultipartUpload":{"DaysAfterInitiation":3}},
    {"ID":"tier-logs","Status":"Enabled","Filter":{"Prefix":"logs/"},
     "Transitions":[{"Days":30,"StorageClass":"STANDARD_IA"},{"Days":120,"StorageClass":"GLACIER_IR"}],
     "NoncurrentVersionExpiration":{"NoncurrentDays":30}}]}'
```

## Choosing between AWS data services: trade-offs


Match the tool to data size, latency, team skills, and cost model.

+ S3 - durable lake storage, not a database; no transactions or row-level updates.
+ Glue - managed Spark/Python ETL; not for very large shuffle-heavy or long-running custom jobs.
+ EMR - large-scale Spark/Hadoop and Spot batch; more operations to own.
+ Lambda - small event-driven tasks; avoid beyond 15 minutes or for large shuffles.
+ Athena - ad-hoc and intermittent SQL over S3; poor for high-concurrency low-latency BI.
+ Redshift - high-concurrency modeled BI; not for tiny point lookups or OLTP.
+ Kinesis - AWS-native streaming; not a task queue and not Kafka-compatible.
+ MSK - best when already on Kafka; you pay for brokers and carry more operations.
+ DynamoDB - high-volume key access at single-digit milliseconds; not ad-hoc analytics.
+ Step Functions - multi-service orchestration; not high-throughput data movement.
+ Lake Formation - fine-grained lake governance; adds complexity for small teams that can start with IAM plus catalog policies.

Cost drivers: bytes scanned (Athena), DPU/worker-hours (Glue, EMR), cluster uptime and concurrency (Redshift), shard-hours and retention (Kinesis), broker-hours (MSK), and storage class plus requests (S3). Security drivers: least-privilege roles, KMS, VPC endpoints, Lake Formation column/row filters, and CloudTrail. Performance drivers: file sizes and formats, partitioning, distribution/sort keys, and shuffle management.

Decide in this order: what the read pattern is (point lookup, scan with predicates, high-concurrency BI), what the latency requirement is (seconds, minutes, hours), and how much operations the team wants to own. Then pick the cheapest engine that satisfies all three, not the most powerful one.

Common mistake: choosing on benchmark results alone. A service the team already operates well usually beats a marginally faster one that adds an unfamiliar failure mode, because total cost includes the on-call rotation.

## Things to remember: key numbers and limits


The numbers below come up constantly in design reviews and interviews. Prices are us-east-1 list prices at the time of writing and shift; the limits are stable.


| Topic | Number / fact | Why it matters |
|---|---|---|
| S3 object | single PUT capped at 5 GiB; multipart part 5 MiB-5 GiB; object max 5 TiB | SDKs switch to multipart above a threshold; abort incomplete uploads with a lifecycle rule |
| S3 listing | `list_objects_v2` returns at most 1,000 keys per call | always use a paginator |
| S3 consistency | strong read-after-write for PUT, overwrite, DELETE, and LIST since Dec 2020 | safe as a lakehouse storage layer; old sleep-and-retry workarounds are obsolete |
| S3 IA / Glacier | 128 KB minimum billable object; 30-day (IA) or 90-day minimum duration | tiny files and short-lived data are the wrong fit; compact first |
| File size target | roughly 128 MB to 1 GB columnar files | Spark runs one task per file; request charges and planning cost scale with file count |
| Athena | about 5 USD per TB scanned, 10 MB minimum per query; engine v3 runs Trino | cost is bytes scanned: Parquet, partitioning, and column pruning are the levers |
| Glue | G.1X = 4 vCPU / 16 GB = 1 DPU; billed per second with a 1-minute minimum | size workers from data volume; watch shuffle spill metrics |
| Lambda | 15 minutes, 10 GB memory, 512 MB writable /tmp | beyond any limit, the job belongs on Glue or ECS |
| Kinesis shard | 1 MB/s or 1,000 records/s in; 2 MB/s out shared by consumers; enhanced fan-out 2 MB/s per consumer | shard math and partition-key cardinality decide throughput |
| Kinesis retention | 24 hours default, up to 365 days | retention is what makes replay possible; SQS has none after ack |
| Firehose | buffers 1-128 MB or 60-900 seconds before delivery | near-real-time, not real-time; buffer settings shape file sizes |
| Redshift | `COPY` parallelizes by input file across slices; `UNLOAD` writes per slice; sort keys drive zone-map skipping | one huge file serializes a load; distribute small dimensions `ALL` |
| DynamoDB | item max 400 KB; partition key hash picks the physical partition | large payloads go to S3; low-cardinality keys create hot partitions |


## Things to remember: semantics to memorize


* **S3 is strongly consistent**, including list — you can write a file and immediately read or list it. Anything repeating the old eventually-consistent answer is outdated.
* **Delivery is at-least-once almost everywhere**: Kinesis, SQS, Lambda triggers, and retried Step Functions states can all deliver a record or run a task twice. Consumers and jobs must be idempotent: overwrite one partition, dedupe on a key, rely on bookmarks.
* **Replay separates queues from streams**: SQS deletes a message on acknowledgement; Kinesis and MSK retain records so a consumer can rewind. If replay matters, it is a stream, not a queue.
* **Athena billing**: DDL is free and a query that fails before reading data is free; you pay for bytes scanned on successful reads. Set a per-query bytes-scanned cutoff in the workgroup.
* **Glue bookmarks track input, not output** — they remember which files were read, not whether the write was correct. A job that rewrites a partition must be idempotent on its own.
* **IAM scoping rules**: object actions need the object ARN (`bucket/*`), `s3:ListBucket` needs the bucket ARN, and wildcard `iam:PassRole` is a privilege-escalation path.
* **`COPY` and `UNLOAD` take an IAM role**, never embedded credentials, and both parallelize across slices — feed them many mid-size files, not one giant one.


## Things to remember: mistakes that keep coming back


Each of these appears in a section above with the fix; this is the checklist form.

* Overwriting the raw zone in place — keep raw immutable and versioned so a transformation bug can be replayed.
* Millions of tiny files — one task and one GET per file; compact to 128 MB-1 GB before tiering or scanning.
* `SELECT *` over raw JSON in Athena — convert to Parquet once (CTAS) and let the dashboard read the aggregate.
* EMR for a ten-minute daily job, or Glue for a multi-hour shuffle-heavy one — match cluster operations to runtime.
* SQS for an analytics stream — no replay after acknowledgement; use Kinesis or MSK.
* Low-cardinality partition keys (status, country, one tenant) on Kinesis or DynamoDB — one hot shard or partition throttles while the rest idle.
* Partitioning a lake table by a high-cardinality column — tens of thousands of tiny partitions; partition by date, bucket by the hot column inside it.
* Distributing a small dimension `EVEN` and joining to a large fact in Redshift — small dimensions belong on `ALL`.
* One huge file into `COPY`, or `UNLOAD ... PARALLEL OFF` — both serialize on a single slice.
* Abandoned multipart uploads — a lifecycle rule with `AbortIncompleteMultipartUpload` stops the silent bill.
* `Resource: "*"` plus `s3:*`, and wildcard `iam:PassRole` — one role per job, exact ARNs, encryption conditions.
* Default `aws/s3` KMS key — it cannot scope who decrypts; use a customer-managed key per data domain.
* Lake Formation registered on a location whose bucket policy still grants direct `s3:GetObject` — governance bypassed completely.
* Non-idempotent orchestration — Step Functions and EventBridge retry, so the Glue job they trigger must tolerate a re-run.


## Real-world use cases on AWS


The four scenarios below put the services together. Each one gives the business context, the data volume, the architecture, and the reason those choices hold up in production. The pattern to notice is that the same primitives keep returning: a durable S3 layer, a catalog or table format over it, compute sized to the workload, and permissions and lifecycle rules applied from the start rather than bolted on later.

### Use case 1: E-commerce clickstream to real-time and batch analytics

Business context: up-to-the-minute conversion dashboards plus next-day cohort analysis. Data volume: ~1.5 TB/day, peaking at 40,000 events/second during promotions.

Architecture: API Gateway -> Kinesis Data Streams (on-demand) -> Lambda enrich -> Firehose -> S3 raw; Glue hourly -> S3 curated Parquet; Athena for ad-hoc, Redshift for BI.

In [ ]:
import base64
import json


def handler(event, context):
    records = []
    for record in event["Records"]:
        payload = json.loads(base64.b64decode(record["kinesis"]["data"]))
        payload["country"] = payload.get("country") or "unknown"
        records.append({"PartitionKey": record["kinesis"]["partitionKey"],
                        "Data": json.dumps(payload) + "\n"})
    return {"records": records}

Why: on-demand Kinesis absorbs promotion spikes without shard math, Firehose delivers cheap micro-batches to S3, Athena serves cheap exploration, and Redshift serves high-concurrency dashboards from the same raw truth.

### Use case 2: CDC from operational databases into a governed warehouse

Business context: near-real-time transaction reporting with strict PII controls. Data volume: ~5 TB/day, ~200 million change records/day from 12 databases.

Architecture: RDS/Aurora -> DMS CDC -> Kinesis/MSK -> Glue streaming -> S3 raw JSON; Glue dedupe/merge -> curated Parquet -> Redshift `COPY`; Lake Formation LF-Tags restrict PII columns.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import Window
from pyspark.sql import functions as F

window = Window.partitionBy("transaction_id").orderBy(F.col("source_lsn").desc())
deduped = cdc.withColumn("rn", F.row_number().over(window)).where(F.col("rn") == 1).drop("rn")

Why: CDC avoids full extracts, Glue dedupes and absorbs schema drift, `COPY` loads files in parallel, and Lake Formation column filters keep raw PII out of analyst queries.

### Use case 3: Cost-controlled log analytics with Athena partition projection

Business context: 13 months of application and CDN logs searchable for incident response under tight cost control. Data volume: ~50 TB in S3, growing ~3 TB/month, queried in bursts.

Architecture: CloudFront/ALB logs -> S3 raw (gzip) -> Athena projection table on `ds`/`hour` -> dashboards, with a 10 GB per-query scan cutoff.

```json
{"Rules":[{"ID":"logs-tier","Status":"Enabled","Filter":{"Prefix":"logs/"},
 "Transitions":[{"Days":30,"StorageClass":"STANDARD_IA"},{"Days":90,"StorageClass":"GLACIER_IR"}]}]}
```

Why: projection removes crawler and `MSCK REPAIR` overhead across tens of thousands of partitions, Parquet conversion of hot prefixes cuts bytes scanned by an order of magnitude, and lifecycle tiers plus a scan cutoff keep the Athena bill predictable.

### Use case 4: A governed multi-team data lake with Lake Formation

Business context: healthcare analytics with data science, finance, and clinical teams sharing one lake under strict column and row restrictions. Data volume: ~20 TB curated across hundreds of tables.

Architecture: producers -> S3 raw (domain prefixes) -> Glue -> S3 curated registered with Lake Formation; consumers use Athena, Redshift Spectrum, or EMR, which honor LF permissions.

```bash
aws lakeformation grant-permissions \
  --principal DataLakePrincipalIdentifier=arn:aws:iam::123456789012:role/clinical-analyst \
  --resource '{"LFTagPolicy":{"ResourceType":"TABLE","Expression":[{"TagKey":"domain","TagValues":["clinical"]},{"TagKey":"classification","TagValues":["public","pii"]}]}}' \
  --permissions SELECT DESCRIBE
```

Why: LF-Tags scale permission management to hundreds of tables without an unmaintainable grant matrix, column exclusions on PHI satisfy compliance, and centrally registered S3 locations stop producers bypassing governance with ad-hoc bucket policies.

## AWS exercises


These exercises use only the Python standard library and turn the cost and sizing arithmetic in this notebook into small, runnable calculations. Each one states the task, a hint, the expected result, and then a code cell that prints and asserts the answer. Run them with the repository virtual environment: `.venv/bin/python`.

The prices used below are the published us-east-1 list prices at the time of writing and are there to make the arithmetic concrete, not to predict a bill. Real invoices add request costs, retrieval fees, data transfer, and taxes, so treat every assert as a check on the formula and the direction of the change.

A useful habit while working through them: write the unit next to every number. Bytes scanned, DPU-hours, shard-hours, and GB-months are different units, and most cost mistakes in production come from mixing two of them.

### Try it 1. Size Athena cost from bytes scanned

**Task.** Athena bills about 5.00 USD per TB scanned, where 1 TB is 1024 GB. Three queries scanned 820 GB, 45.5 GB, and 12.25 GB. Compute the total scanned TB, the cost of each query, and the total cost.

**Hint.** Convert GB to TB by dividing by 1024, then multiply by 5.00. Keep the unrounded values and round only for printing.

**Expected result.** About `0.857` TB scanned and a total cost of about `4.29` USD.

In [ ]:
GB_PER_TB = 1024
PRICE_PER_TB = 5.00
scans_gb = [820.0, 45.5, 12.25]


def cost(gb):
    return gb / GB_PER_TB * PRICE_PER_TB


per_query = [cost(gb) for gb in scans_gb]
for gb, usd in zip(scans_gb, per_query):
    print(f"{gb:8.2f} GB -> {usd:.4f} USD")

total_gb = sum(scans_gb)
total_tb = total_gb / GB_PER_TB
total_usd = sum(per_query)
print(f"total scanned = {total_gb:.2f} GB = {total_tb:.3f} TB")
print(f"total cost = {total_usd:.4f} USD")

assert round(total_tb, 3) == 0.857
assert round(total_usd, 2) == 4.29


### Try it 2. Choose a partition scheme from query filters

**Task.** Given how often each column appears in the `WHERE` clause, choose a two-column partition scheme. Keep only columns that appear in at least 10 queries, order them by filter frequency (ties broken alphabetically), and take the first two. Then compute how many partitions the scheme creates and how many a single-day query with one region scans.

**Hint.** Sort by `(-count, name)` so the most-filtered column comes first. Partitions are the product of the distinct values of the chosen columns.

**Expected result.** The scheme is `('event_date', 'region')` with `7,300` partitions, and one day plus one region scans `20` partitions instead of `7,300`.

In [ ]:
filter_counts = {
    "event_date": 100,
    "region": 40,
    "country": 12,
    "customer_id": 2,
}
cardinality = {"event_date": 365, "region": 20, "country": 12, "customer_id": 500000}

candidates = [name for name, count in filter_counts.items() if count >= 10]
ranked = sorted(candidates, key=lambda name: (-filter_counts[name], name))
scheme = tuple(ranked[:2])

partitions = 1
for column in scheme:
    partitions *= cardinality[column]
day_with_region = cardinality["region"]

print("scheme:", scheme)
print("partitions:", f"{partitions:,}")
print("scanned for one day and one region:", day_with_region)

assert scheme == ("event_date", "region")
assert partitions == 7300
assert day_with_region == 20


### Try it 3. Compute compaction savings from file counts

**Task.** A prefix holds 240,000 small Parquet files of 4 MB each. S3 GET requests cost 0.0004 USD per 1,000 requests. Compute the total data size, the number of files after compacting to 256 MB files, the file-count reduction, and the request cost of one full scan before and after.

**Hint.** `files_after = total_mb / 256`. Request cost is `files / 1000 * 0.0004`. Use `math.ceil` for the file count so a partial file still counts.

**Expected result.** `3,750` files after compaction, a `64x` reduction, and a scan request cost falling from `0.0960` to `0.0015` USD.

In [ ]:
import math

files_before = 240_000
file_size_mb = 4
target_mb = 256
request_price = 0.0004  # USD per 1,000 GET requests

total_mb = files_before * file_size_mb
files_after = math.ceil(total_mb / target_mb)
reduction = files_before / files_after

cost_before = files_before / 1000 * request_price
cost_after = files_after / 1000 * request_price

print(f"total size = {total_mb:,} MB = {total_mb / 1024:.1f} GB")
print(f"files after compaction = {files_after:,}")
print(f"file-count reduction = {reduction:.1f}x")
print(f"scan request cost = {cost_before:.4f} -> {cost_after:.4f} USD")

assert files_after == 3750
assert reduction == 64.0
assert round(cost_before, 4) == 0.096
assert round(cost_after, 4) == 0.0015


### Try it 4. Size Glue workers from data volume

**Task.** A shuffle-heavy Glue job reads 1.2 TB of Parquet. Use a planning rule of 60 percent of a worker's memory as usable capacity and a G.1X worker (16 GB), and assume the job needs to hold the input once. Compute the worker count and the cost of a 30 minute run at 0.44 USD per DPU-hour, where one G.1X worker is one DPU.

**Hint.** Usable GB per worker is `16 * 0.6 = 9.6`. Workers are `ceil(total_gb / 9.6)`. Cost is `workers * hours * 0.44`.

**Expected result.** `128` G.1X workers and a run cost of `28.16` USD for 30 minutes.

In [ ]:
import math

input_tb = 1.2
worker_gb = 16
usable_fraction = 0.6
price_per_dpu_hour = 0.44
runtime_hours = 0.5

input_gb = input_tb * 1024
usable_gb = worker_gb * usable_fraction
workers = math.ceil(input_gb / usable_gb)
cost = workers * runtime_hours * price_per_dpu_hour

print(f"input = {input_gb:.1f} GB, usable per worker = {usable_gb:.1f} GB")
print(f"workers = {workers}")
print(f"cost for {runtime_hours:g} h = {cost:.2f} USD")

assert workers == 128
assert round(cost, 2) == 28.16


### Try it 5. Size Kinesis shards from records per second and payload

**Task.** A producer sends 20,000 records per second with an average payload of 1.2 KB. One shard accepts 1,000 records per second or 1 MB per second, whichever limit is reached first. Compute the shards needed for each limit, take the larger, add 30 percent headroom, and price the stream for a 730 hour month at 0.015 USD per shard-hour.

**Hint.** Records limit is `rps / 1000`. Bytes limit is `rps * payload_kb / 1024` MB per second, rounded up. Headroom applies to the larger of the two.

**Expected result.** `20` shards from the record limit, `24` from the byte limit, `32` shards after headroom, and a monthly shard cost of about `350.40` USD.

In [ ]:
import math

rps = 20_000
payload_kb = 1.2
records_per_shard = 1000
kb_per_shard = 1024
headroom = 1.30
shard_price_hour = 0.015
hours_per_month = 730

shards_records = math.ceil(rps / records_per_shard)
mb_per_second = rps * payload_kb / 1024
shards_bytes = math.ceil(mb_per_second / 1)
required = max(shards_records, shards_bytes)
provisioned = math.ceil(required * headroom)
monthly = provisioned * shard_price_hour * hours_per_month

print(f"throughput = {mb_per_second:.2f} MB/s")
print(f"shards: records={shards_records} bytes={shards_bytes} required={required}")
print(f"with headroom = {provisioned} shards")
print(f"monthly shard cost = {monthly:.2f} USD")

assert shards_records == 20
assert shards_bytes == 24
assert provisioned == 32
assert round(monthly, 2) == 350.40


### Try it 6. Compare S3 storage-class cost over a year

**Task.** A bucket holds 100 TB of cold data. Compare 12 months of S3 Standard at 0.023 USD per GB-month with Standard-IA at 0.0125 USD per GB-month, and report the annual saving. Use 1 TB equals 1024 GB.

**Hint.** Annual cost is `gb * price * 12`. Saving is the difference between the two annual figures.

**Expected result.** Standard costs `28,262.40` USD a year, Standard-IA costs `15,360.00`, and the saving is `12,902.40` USD.

In [ ]:
TB = 1024
size_gb = 100 * TB
standard_price = 0.023
ia_price = 0.0125
months = 12

standard_year = size_gb * standard_price * months
ia_year = size_gb * ia_price * months
saving = standard_year - ia_year

print(f"size = {size_gb:,} GB")
print(f"Standard    = {standard_year:,.2f} USD/year")
print(f"Standard-IA = {ia_year:,.2f} USD/year")
print(f"saving      = {saving:,.2f} USD/year")

assert round(standard_year, 2) == 28262.40
assert round(ia_year, 2) == 15360.00
assert round(saving, 2) == 12902.40


### Try it 7. Plan an EMR fleet with Spot

**Task.** A two hour job needs 1 driver and 8 core nodes on On-Demand at 0.504 USD per hour, plus 16 task nodes on Spot at 0.15 USD per hour. Compute the cost of the mixed fleet, the cost if every node were On-Demand, and the percentage saved.

**Hint.** Only task nodes go on Spot; the driver and core nodes must survive an interruption. Multiply node-hours by the hourly rate, then by the runtime.

**Expected result.** The mixed fleet costs `13.872` USD against `25.20` On-Demand, a saving of about `45.0` percent.

In [ ]:
on_demand_rate = 0.504
spot_rate = 0.15
runtime_hours = 2.0

on_demand_nodes = 1 + 8       # driver + core
spot_nodes = 16               # task
all_nodes = on_demand_nodes + spot_nodes

mixed = (on_demand_nodes * on_demand_rate + spot_nodes * spot_rate) * runtime_hours
on_demand_only = all_nodes * on_demand_rate * runtime_hours
saved_pct = (on_demand_only - mixed) / on_demand_only * 100

print(f"mixed fleet  = {mixed:.3f} USD")
print(f"all On-Demand = {on_demand_only:.2f} USD")
print(f"saved = {saved_pct:.1f}%")

assert round(mixed, 3) == 13.872
assert round(on_demand_only, 2) == 25.20
assert round(saved_pct, 1) == 45.0


### Try it 8. Estimate Glue cost with and without job bookmarks

**Task.** A nightly job without bookmarks reprocesses all 400 GB in 45 minutes on 20 G.1X workers. With bookmarks enabled it processes 40 GB of new data in 6 minutes on 10 workers. At 0.44 USD per DPU-hour, compute both run costs, the daily saving, and the annual saving.

**Hint.** Convert minutes to hours, then cost is `workers * hours * 0.44`. The annual saving multiplies the daily saving by 365.

**Expected result.** The full run costs `6.60` USD, the incremental run costs `0.44` USD, and the annual saving is `2,248.40` USD.

In [ ]:
price_per_dpu_hour = 0.44


def run_cost(workers, minutes):
    return workers * (minutes / 60) * price_per_dpu_hour


full = run_cost(20, 45)
incremental = run_cost(10, 6)
daily_saving = full - incremental
annual_saving = daily_saving * 365

print(f"full run        = {full:.2f} USD")
print(f"incremental run = {incremental:.2f} USD")
print(f"daily saving    = {daily_saving:.2f} USD")
print(f"annual saving   = {annual_saving:.2f} USD")

assert round(full, 2) == 6.60
assert round(incremental, 2) == 0.44
assert round(annual_saving, 2) == 2248.40
